# Experiment 3 — Stratified vs. Global Stopping (the Cinder Failure)

**Question** (experiment proposal, Exp 3): does a *global-only* "have I seen enough?" certificate pass while one
small group is silently wrong — and does *per-group* stopping catch it?

**Claim under test** — **C3**: every `GROUP BY` group gets its own guarantee, *simultaneously*; a global bound
can pass while one small group is badly wrong (design paper §3.3 "*A global bound does not suffice*",
Theorem 2, and the §3.4 worked run, where a global-only agent ships a table "*silently 25% wrong for Cinder*").

**The idea in one paragraph.** The agent hunts the open web for companies' funding rounds while the
authoritative registry (SEC EDGAR) is *deliberately hidden from it*; the true answer key is built from that
same registry, so we can grade what the agent found without the agent ever seeing the answers. The trap this
experiment sets is *skew*: each query asks about a small **portfolio** of companies at once — several
well-covered ones plus one tiny one, the "Cinder" of the group, holding only 2–3 of the portfolio's true
records. A **global** stopping rule pools everything into one number: with, say, 25 records total and 2 of
them Cinder's, missing *every* Cinder record costs under a tenth of the pooled mass, so the pooled test can
declare "done" with a straight face while one company's history is entirely absent. The **stratified** rule
runs the same test *per company* and may only stop when every company passes at once (Theorem 2). We measure,
for both rules, what fraction of *each* company's true rounds had actually been found at the moment that rule
said "done".

**Setup (one line).** 6 skew-by-construction portfolios drawn from the 34 **calibration** entities of the
frozen `formd_v2` cohort; one real end-to-end run per portfolio (real Serper searches, real LLM extraction
behind the fitted conformal gate, real ER with live adjudication, registry denied); both stopping times are
then read *off the same recorded trajectory* and graded per company against the withheld registry.

**Expected result.** Global: the certificate fires early, the *pooled* recall at that moment is fine
(≥ 1−ε — the pooled promise is kept!), but the small company is materially short — the failure is *silent*
precisely because the global number looks healthy. Stratified: when it certifies, every company meets its
target simultaneously (violations ≤ δ); the price is extra steps and spend, which we also report. If the
global rule rarely hurts anyone, stratification solves a rare problem — that would be an honest, reportable
bad outcome (proposal, Exp 3).

**Why it matters.** Per-group semantics is what makes this a `GROUP BY` paper rather than a recall paper;
this reproduces the paper's worked run (§3.4) at scale on real web data.

**Design changes vs. the proposal** (the proposal invites improvements; both are pre-registered here, before
any live run — flag both to the supervisor in the next update):

1. **One live run per portfolio, not two.** The stop test is evaluated only at the *end* of each step and can
   only end the run — neither ε nor the stratified/global choice ever influences which search is issued next
   (the same ε-free-path fact Experiment 2 already relies on). So a single run under the stratified rule
   *contains* the global run as a prefix, and the global stopping time can be read off the same trajectory —
   **exactly**, at half the cost, with no noise from the search engine returning different results on a
   second pass. Every comparison below is therefore *paired*: same searches, same pages, same extractions.
2. **Two pre-registered test variants.** The production stop test includes the anytime-valid radius ψ. The
   paper itself notes that at small stratum sizes ψ dwarfs ε ("*the statistical certificate may be
   unattainable at any sane budget — the truthful state of affairs, not a defect*", Thm 2 remark) and that
   its own worked run (Table 2) traces the stop at the **estimate level** (Û against ε plus the frontier
   conjunct, no ψ in the table). At this cohort's scale (2–9 true records per company) the ψ-inclusive test
   is expected to fire for *neither* arm — we run and report it anyway (variant **full**), and draw the
   stratified-vs-global contrast in the worked run's own regime (variant **estimate**: Û < ε ∧ frontier
   cold). This mirrors Exp 2's Figure 1, which plots the same ψ-on/ψ-off pair.

**Outputs**: `figures/exp3_*.pdf` (paper figures), `data/experiments/exp3/exp3_*.csv` (paper tables).

In [6]:
# --- imports, repo root, and the PRE-REGISTERED parameters -------------------
# Everything an arm of this experiment depends on is fixed HERE, before any
# result is looked at (repo principle: pre-register grading decisions).
from __future__ import annotations

import json
import math
import os
import sys
import inspect
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# repo root: the notebook lives in notebooks/, the package one level up
ROOT = Path.cwd()
if not (ROOT / "webagg").exists():
    ROOT = ROOT.parent
assert (ROOT / "webagg").exists(), f"cannot find the webagg package from {Path.cwd()}"
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)   # webagg opens prompts/ and data/ relative to the working dir

from webagg import config, pipeline                       # the real pipeline
from webagg.storage import (get_session, MeasurementRow,  # the measurement spine
                            MentionRow, SourceRow, FrontierFormulationRow)
from webagg.certify import REGISTRY_DENY
from webagg.formd import load_truth_cohort
from webagg.frontier import normalize_surface, FrontierState
from webagg.risk_control import match_to_truth, default_truth_key

# ---- pre-registered experiment parameters -----------------------------------
COHORT      = "formd_v2"
EPS         = 0.10                 # the one promise level: every group >= 90%
DELTA       = config.DELTA_M       # psi confidence budget (paper delta_M)
ETA         = config.ETA           # hot-frontier threshold (conjunct ii)
MAX_STEPS   = 120                  # 2x the single-entity A3 cap: 4 companies/query
BUDGET_USD  = 10.0                 # per-portfolio SEARCH spend cap (real)
QUERY_ATTRS = {"amount", "date"}   # same pair the certification runs use
AMOUNT_TOL  = config.GRADING_AMOUNT_TOL   # 5% -- pre-registered A3 attempt-#3
Y_CAP, BETA = config.Y_CAP, config.BETA   # must match the live loop's constants

# portfolio construction rule (deterministic, from the FROZEN truth only):
SMALL_MAX_RECORDS = 3   # a "Cinder" holds <= 3 true records
PORTFOLIO_SIZE    = 4   # 1 small + 3 large companies per query

# PRE-REGISTERED EXCLUSION: the 7 generic-name-starved calibration entities.
# The A3 fidelity diagnosis (2026-09-07) found their reference runs surface
# ZERO records under their names -- including them would test findability
# (Theorem 2's delta_G term: undiscovered strata), not stopping (its eps_g
# term). Excluded from portfolio membership; disclosed in the write-up.
STARVED = {
    "cik0001712223",  # Prenda
    "cik0001783848",  # Omsom
    "cik0001832759",  # Settle
    "cik0001959571",  # Magic AI
    "cik0001982311",  # Unstructured
    "cik0001783955",  # System Initiative
    "cik0001799591",  # Equip Health
}

# the ONE query template all portfolio runs share (a per-portfolio prompt
# tweak would be tuning). Keeps the A3 attempt-#3 universe scoping (equity
# only, exclude debt/IPO/secondary) and adds the GROUP BY semantics.
def portfolio_query(names: list[str]) -> str:
    listed = ", ".join(names[:-1]) + f", and {names[-1]}"
    return (f"total equity funding raised in private funding rounds by each of "
            f"the companies {listed}, reported separately for each company, "
            f"excluding debt financing, IPO proceeds, and share sales")

# the live phase costs real money; nothing live runs until this is True
RUN_LIVE = True

EXP_DIR = config.DATA_DIR / "experiments" / "exp3"
FIG_DIR = ROOT / "figures"
EXP_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

ARM_COLOR = {"global": "#D55E00", "stratified": "#0072B2"}   # colorblind-safe
plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"repo root : {ROOT}")
print(f"promise   : every group recall >= {1-EPS:.2f} (eps={EPS}, delta={DELTA})")
print(f"budget    : ${BUDGET_USD:.2f} search spend / portfolio, max {MAX_STEPS} steps")
print(f"variants  : 'full' (U_hat + psi, production) and 'estimate' (U_hat only,")
print(f"            the paper's worked-run regime) -- both pre-registered")

repo root : C:\Users\wangz\PycharmProjects\webagg
promise   : every group recall >= 0.90 (eps=0.1, delta=0.1)
budget    : $10.00 search spend / portfolio, max 120 steps
variants  : 'full' (U_hat + psi, production) and 'estimate' (U_hat only,
            the paper's worked-run regime) -- both pre-registered


## Preflight — refuse to run degraded

Each check guards one seam that could otherwise degrade *silently* and quietly turn a "real" experiment into
a fake one. A failed check stops the notebook with instructions; nothing downstream runs on a broken seam.

In [7]:
# --- preflight: every seam checked LOUDLY, before a dollar is spent ----------
failures: list[str] = []
def check(name: str, ok: bool, fix: str):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        failures.append(f"{name}\n         fix: {fix}")

print("preflight:")

# 1. the Exp-3 instrumentation patch is in pipeline.run_query
src = inspect.getsource(pipeline.run_query)
check("Exp-3 instrumentation patch (U_hat_global logged per step)",
      '"U_hat_global"' in src,
      "apply the patch from the Prerequisites cell to webagg/pipeline.py, "
      "then re-run pytest (206 tests must pass)")
# ... and the Exp-2 per-stratum fields it builds on
check("Exp-2 per-stratum fields (hot/certified/spent/formulation_id)",
      all(f'"{k}"' in src for k in ("hot", "certified", "spent", "formulation_id")),
      "the Exp-2 instrumentation patch is missing from pipeline.run_query")

# 2. frozen cohort manifest with the calibration split
cohort_dir = config.GROUND_TRUTH_DIR / COHORT
manifest_path = cohort_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
cal_ids = (manifest or {}).get("split", {}).get("calibration", [])
check(f"cohort manifest with calibration split ({len(cal_ids)} entities)",
      bool(cal_ids), f"build the {COHORT} cohort (scripts/build_truth.py)")

# 3. answer keys load for every calibration entity
try:
    truths = load_truth_cohort(cohort_dir)
    missing = [e for e in cal_ids if e not in truths]
    check("answer keys present for all calibration entities", not missing,
          f"missing truth JSONs: {missing[:5]}")
except Exception as exc:
    truths = {}
    check("answer keys load", False, f"load_truth_cohort failed: {exc}")

# 4. the A3 reference-run ledger (source of the open-web COMMON names --
#    the Instacart lesson: EDGAR files legal names the press never uses)
a3_index_path = config.FIDELITY_CERT_DIR / f"{COHORT}.runs.json"
a3_index = json.loads(a3_index_path.read_text()) if a3_index_path.exists() else {}
check("A3 reference-run ledger (common names)", bool(a3_index),
      f"missing {a3_index_path}")
NAMES = {eid: info.get("entity_name") for eid, info in a3_index.items()}

# 5. seam A1: the conformal gate has a real calibration set
from webagg.calibration import load_calibration_set
cal_set = load_calibration_set(config.CALIBRATION_SET)
check(f"A1 conformal gate calibration set ({len(cal_set or [])} examples)",
      bool(cal_set), "seam A1 must be closed (gate would run accept-all)")

# 6. seam A2: the ER matcher is fitted (not cold-start)
try:
    from webagg.er_pairs import load_fitted_matcher
    _m = load_fitted_matcher()
    check(f"A2 fitted ER matcher (alpha={getattr(_m, 'alpha', None)})",
          getattr(_m, "alpha", None) is not None,
          "seam A2 must be closed (matcher is cold)")
except Exception as exc:
    check("A2 fitted ER matcher", False, f"load_fitted_matcher failed: {exc}")

# 7. live keys
check("SERPER_API_KEY present", bool(config.SERPER_API_KEY), "add to .env")
check("LLM provider key present", bool(os.environ.get("OPENAI_API_KEY")),
      "add OPENAI_API_KEY to .env")

if failures:
    raise RuntimeError("preflight failed:\n" + "\n".join(failures))
print("preflight clean.")

preflight:
  [PASS] Exp-3 instrumentation patch (U_hat_global logged per step)
  [PASS] Exp-2 per-stratum fields (hot/certified/spent/formulation_id)
  [PASS] cohort manifest with calibration split (34 entities)
  [PASS] answer keys present for all calibration entities
  [PASS] A3 reference-run ledger (common names)
  [PASS] A1 conformal gate calibration set (111 examples)
[er] matcher fitted on 460 labeled pairs (148 same / 312 different); alpha=0.3739
  [PASS] A2 fitted ER matcher (alpha=0.3739130434782609)
  [PASS] SERPER_API_KEY present
  [PASS] LLM provider key present
preflight clean.


## Step 1 — build the skewed portfolios (deterministic, from the frozen truth only)

Each portfolio is one `GROUP BY` query over four companies: **one small** ("Cinder": ≤ 3 true records in the
registry) and **three large**. The construction is a fixed rule over the frozen answer key — no discretion,
nothing tuned after seeing results: small companies (excluding the pre-registered starved seven) are sorted
by (true record count, entity id) and each anchors one portfolio; the large companies are sorted by
(−count, entity id) and dealt across portfolios in snake order so total sizes stay balanced. Leftover large
entities are simply unused (printed below). The **skew** column is the small company's share of its
portfolio's true records — the quantity the global test is blind to.

In [8]:
# --- deterministic portfolio construction ------------------------------------
counts = {e: truths[e].true_count for e in cal_ids}
eligible = [e for e in cal_ids if e not in STARVED]

small = sorted([e for e in eligible if counts[e] <= SMALL_MAX_RECORDS],
               key=lambda e: (counts[e], e))
large = sorted([e for e in eligible if counts[e] > SMALL_MAX_RECORDS],
               key=lambda e: (-counts[e], e))
n_pf = len(small)
need = n_pf * (PORTFOLIO_SIZE - 1)
assert len(large) >= need, f"only {len(large)} large entities for {need} slots"

deal = [[] for _ in range(n_pf)]                    # snake deal: balanced totals
for i, eid in enumerate(large[:need]):
    rnd, pos = divmod(i, n_pf)
    deal[pos if rnd % 2 == 0 else n_pf - 1 - pos].append(eid)

portfolios: dict[str, dict] = {}
for k, s in enumerate(small, start=1):
    pid = f"p{k}"
    members = [s] + deal[k - 1]
    portfolios[pid] = {
        "pid": pid, "small": s, "members": members,
        "names": {e: NAMES.get(e) or manifest["entities"][e]["entity_name"]
                  for e in members},
        "truth_counts": {e: counts[e] for e in members},
    }

rows = []
for pid, pf in portfolios.items():
    tot = sum(pf["truth_counts"].values())
    rows.append(dict(
        portfolio=pid,
        small=pf["names"][pf["small"]],
        small_records=pf["truth_counts"][pf["small"]],
        members=", ".join(pf["names"][e] for e in pf["members"]),
        total_records=tot,
        skew=round(pf["truth_counts"][pf["small"]] / tot, 3)))
setup = pd.DataFrame(rows).set_index("portfolio")
print(f"{n_pf} portfolios; unused large entities: "
      f"{[NAMES.get(e, e) for e in large[need:]]}")
setup

6 portfolios; unused large entities: ['Arris Composites', 'Magic Spoon', 'Apptronik']


,small,small_records,members,total_records,skew
portfolio,,,,,
p1,Loft Labs,2,"Loft Labs, Hinge Health, Stoke Space, MarginEdge",22,0.091
p2,Squarespace,3,"Squarespace, Aledade, Billd, Eleanor Health",22,0.136
p3,Arable,3,"Arable, Stord, Expel, Ambi Robotics",22,0.136
p4,AirGarage,3,"AirGarage, AMP Robotics, Illumio, Nth Cycle",21,0.143
p5,Abodu,3,"Abodu, Tive, Hadrian, Element Science",21,0.143
p6,Ebb Carbon,3,"Ebb Carbon, Crown Affair, Supabase, Zap Energy",21,0.143


## Why one live run yields both arms (and what the run costs)

Think of the run as a **flight recorder**. At every step the loop logs, per company-stratum, exactly what any
stop test could have seen at that moment — the unseen-mass estimate $\hat U_g$, its radius $\psi_g$, whether
a promising search was still pending (*hot*), certificates, money spent — and (the Exp-3 patch) the same
statistic **pooled across all companies**, which is what a global-only agent would watch. Because ε and the
choice of test are read *only* inside the end-of-step stop check, they can end the run but never steer it:
the searches issued, pages fetched, and records extracted are identical whichever rule is in force. So the
recorded trajectory *is* both runs — the global arm is the prefix up to the first step its pooled test
passes, the stratified arm the prefix up to the first step every company passes. Each portfolio becomes a
perfectly paired comparison, immune to the search engine returning different pages on a second pass.

The live phase costs real money. The cell below projects the bill from the **measured** per-step cost of the
34 A3 reference runs on this machine (tokens by model, straight from their run DBs), plus a parameterized
adjudication estimate, and nothing live runs until `RUN_LIVE = True`.

In [9]:
# --- spend projection from the A3 reference-run DBs (measured, not guessed) --
# EDIT to current OpenAI list prices before trusting the dollar figure:
PRICE_PER_MTOK = {config.MODEL_STRONG: {"in": 1.25, "out": 10.00},   # gpt-5
                  config.MODEL_CHEAP:  {"in": 0.05, "out":  0.40}}   # gpt-5-nano
ADJ_PAIRS_PER_ENTITY = 175    # measured offline for A3 (scripts/
                              # estimate_adjudication_load.py: 5,946 band
                              # pairs / 34 entities); resolve-time band pairs
ADJ_TOKS_PER_PAIR = (900, 60) # (input, output) tokens per adjudication, approx

tok = {m: {"in": 0.0, "out": 0.0} for m in PRICE_PER_MTOK}
ref_steps, n_ref = 0, 0
for eid, info in a3_index.items():
    db = Path(info["db"])
    if not db.exists():
        continue
    session = get_session(str(db))
    try:
        n_ref += 1
        ref_steps += (session.query(MeasurementRow.step)
                      .filter(MeasurementRow.run_id == info["run_id"])
                      .order_by(MeasurementRow.step.desc()).first() or (0,))[0]
        for r in (session.query(MeasurementRow)
                  .filter(MeasurementRow.run_id == info["run_id"],
                          MeasurementRow.metric == "llm_call")):
            e = r.extra or {}
            m = e.get("model")
            if m in tok:
                tok[m]["in"] += e.get("input_tokens", 0)
                tok[m]["out"] += e.get("output_tokens", 0)
    finally:
        engine = session.get_bind(); session.close(); engine.dispose()

n_members = sum(len(pf["members"]) for pf in portfolios.values())
if n_ref:
    per_step_usd = sum(tok[m][d] / ref_steps * PRICE_PER_MTOK[m][d[:0] or d] / 1e6
                       for m in tok for d in ("in", "out"))
    disc = per_step_usd * MAX_STEPS * len(portfolios)
    print(f"measured over {n_ref} reference runs, {ref_steps} steps:")
    for m in tok:
        print(f"  {m:>12}: {tok[m]['in']/1e6:.2f}M in / {tok[m]['out']/1e6:.2f}M out tokens")
    print(f"  -> ${per_step_usd:.4f}/step; discovery cap for "
          f"{len(portfolios)} x {MAX_STEPS} steps ~= ${disc:.2f} (upper bound; "
          f"runs may stop earlier)")
else:
    per_step_usd = 0.02
    disc = per_step_usd * MAX_STEPS * len(portfolios)
    print("NOTE: no A3 reference-run DBs found on this machine -- falling back "
          f"to a flat ${per_step_usd:.2f}/step assumption.")
adj_pairs = ADJ_PAIRS_PER_ENTITY * n_members
adj = adj_pairs * (ADJ_TOKS_PER_PAIR[0] * PRICE_PER_MTOK[config.MODEL_CHEAP]["in"]
                   + ADJ_TOKS_PER_PAIR[1] * PRICE_PER_MTOK[config.MODEL_CHEAP]["out"]) / 1e6
search = 0.02 * MAX_STEPS * len(portfolios)
print(f"adjudication (resolve): ~{adj_pairs} band pairs ~= ${adj:.2f}")
print(f"search (Serper):        <= ${search:.2f}")
print(f"PROJECTED TOTAL:        ~= ${disc + adj + search:.2f} "
      f"for {len(portfolios)} portfolios")
if not RUN_LIVE:
    print("\nRUN_LIVE = False -- the live cell below will be skipped. Set "
          "RUN_LIVE = True in the parameters cell to spend the budget.")

NOTE: no A3 reference-run DBs found on this machine -- falling back to a flat $0.02/step assumption.
adjudication (resolve): ~4200 band pairs ~= $0.29
search (Serper):        <= $14.40
PROJECTED TOTAL:        ~= $29.09 for 6 portfolios


In [ ]:
# --- STEP 2 (LIVE): one real end-to-end run per portfolio --------------------
# The production path throughout: real Serper, real LLM extraction behind the
# fitted gate, registry DENIED, then the real ER matcher with live LLM band
# adjudication. Idempotent + ledger-resumable: finished portfolios are skipped
# LOUDLY; a crashed run resumes at the next portfolio on re-execution.
INDEX_PATH = EXP_DIR / "exp3_runs_index.json"
runs_index = json.loads(INDEX_PATH.read_text()) if INDEX_PATH.exists() else {}

def _cv_plain(cv):
    """CorroboratedValue -> plain dict of exactly what grading reads."""
    if cv is None:
        return None
    return {"value": getattr(cv, "value", None),
            "value_num": getattr(cv, "value_num", None)}

for pid, pf in portfolios.items():
    run_id = f"exp3_{pid}"
    db = config.RUNS_DIR / f"{run_id}.sqlite"
    rec_path = EXP_DIR / f"records_{pid}.json"
    if pid in runs_index and db.exists() and rec_path.exists():
        print(f"[exp3] {pid}: run exists -- skip")
        continue
    if not RUN_LIVE:
        print(f"[exp3] {pid}: RUN_LIVE=False -- skip (not yet run)")
        continue

    names = [pf["names"][e] for e in pf["members"]]
    q = portfolio_query(names)
    print(f"[exp3] {pid}: LIVE run over {names}\n        query={q!r}")
    # discovery under the PRODUCTION (stratified, psi-inclusive) rule -- the
    # later-stopping arm, so the trajectory contains the global arm as a prefix
    state, session = pipeline.run_query(
        q, run_id=run_id, eps=EPS, delta=DELTA, eta=ETA,
        max_steps=MAX_STEPS, budget_usd=BUDGET_USD,
        query_attributes=QUERY_ATTRS, deny=REGISTRY_DENY)
    try:
        # resolution on the FULL pool: real fitted matcher + live adjudicator,
        # post-ER checksum revalidation via state (design Sec. 14.1)
        result = pipeline.resolve_and_aggregate(
            session, run_id=run_id, query_attributes=QUERY_ATTRS,
            aggregate_attr="amount", eps=EPS, state=state,
            mode="open_web", delta_M=DELTA, max_steps=MAX_STEPS)
    finally:
        engine = session.get_bind()
        session.close()
        engine.dispose()            # Windows: release the sqlite handle

    # persist the REAL resolved records verbatim (resumability, offline grading)
    plain = [{"entity_id": r["entity_id"], "record_kind": r["record_kind"],
              "attributes": {a: _cv_plain(cv)
                             for a, cv in (r["attributes"] or {}).items()},
              # LOUD (repo discipline): a record without its provenance handle
              # must crash here, not silently grade to recall 0 downstream
              "contributing_mentions": list(r["contributing_mentions"])}
             for r in result["records"]]
    rec_path.write_text(json.dumps(plain, indent=1))
    runs_index[pid] = {"db": str(db), "run_id": run_id, "query": q,
                       "members": pf["members"], "small": pf["small"],
                       "names": pf["names"], "eps": EPS, "delta": DELTA,
                       "eta": ETA, "max_steps": MAX_STEPS,
                       "budget_usd": BUDGET_USD, "deny": list(REGISTRY_DENY),
                       "n_records": len(plain)}
    INDEX_PATH.write_text(json.dumps(runs_index, indent=2, sort_keys=True))
    print(f"[exp3] {pid}: done -- {len(plain)} resolved records")

done = [p for p in portfolios if p in runs_index
        and Path(runs_index[p]["db"]).exists()
        and (EXP_DIR / f"records_{p}.json").exists()]
print(f"\nlive phase: {len(done)}/{len(portfolios)} portfolios available")
assert done, "no completed portfolio runs yet: set RUN_LIVE=True and re-run"

[exp3] p1: LIVE run over ['Loft Labs', 'Hinge Health', 'Stoke Space', 'MarginEdge']
        query='total equity funding raised in private funding rounds by each of the companies Loft Labs, Hinge Health, Stoke Space, and MarginEdge, reported separately for each company, excluding debt financing, IPO proceeds, and share sales'


## Step 3 (offline) — load the flight recorder

Every step of every run left one row per company-stratum (the Exp-2 fields) **plus** one pooled row (the
Exp-3 patch). All analysis from here on reads only this ledger and the run DBs — **no new fetches, no new
LLM calls, no new spend**.

In [ ]:
# --- load per-step ledgers: per-stratum rows + the pooled row + actual stop --
def load_ledgers(db_path: str, run_id: str):
    session = get_session(db_path)
    try:
        rows = (session.query(MeasurementRow)
                .filter(MeasurementRow.run_id == run_id,
                        MeasurementRow.metric == "U_hat")
                .order_by(MeasurementRow.step).all())
        per = [dict(step=r.step, stratum=r.stratum, U=r.value,
                    **{k: (r.extra or {}).get(k)
                       for k in ("psi", "N", "f1", "f2", "hot", "certified",
                                 "spent", "claimed_count", "formulation_id")})
               for r in rows]
        rows_g = (session.query(MeasurementRow)
                  .filter(MeasurementRow.run_id == run_id,
                          MeasurementRow.metric == "U_hat_global")
                  .order_by(MeasurementRow.step).all())
        glob = [dict(step=r.step, Ug=r.value,
                     **{k: (r.extra or {}).get(k)
                        for k in ("N", "f1", "f2", "T", "n_strata",
                                  "n_uncertified", "spent")})
                for r in rows_g]
        stop_row = (session.query(MeasurementRow)
                    .filter(MeasurementRow.run_id == run_id,
                            MeasurementRow.metric == "stop")
                    .order_by(MeasurementRow.step.desc()).first())
        stop = (stop_row.step, (stop_row.extra or {}).get("reason")) \
            if stop_row else (None, None)
    finally:
        engine = session.get_bind(); session.close(); engine.dispose()
    return pd.DataFrame(per), pd.DataFrame(glob), stop

step_logs, glob_logs, actual_stops = {}, {}, {}
for pid in done:
    step_logs[pid], glob_logs[pid], actual_stops[pid] = load_ledgers(
        runs_index[pid]["db"], runs_index[pid]["run_id"])
    df, dfg = step_logs[pid], glob_logs[pid]
    print(f"{pid}: {dfg['step'].nunique():>3} steps, "
          f"{df['stratum'].nunique():>2} strata, "
          f"actual stop = {actual_stops[pid]}")
assert all(len(glob_logs[p]) for p in done), \
    "a run logged no U_hat_global rows: was the Exp-3 patch applied BEFORE it ran?"


## Step 4 (offline) — replay both stopping rules on the recorded trajectory

Both replays mirror the live loop's exact evaluation order (budget first, then the certificate test; the
cold-start guard; checksum-closed strata exempt; the cardinality brake can only forbid stopping):

- **Stratified** (the production rule, `pipeline.all_strata_pass`): stop at the first step where *every*
  not-yet-certified company-stratum has (i) $\hat U_g + \psi_g < \varepsilon$ and (ii) no hot pending search.
  This is a line-for-line mirror of Exp 2's replay, so the same self-test applies: at the live parameters it
  must reproduce the run's own stop exactly.
- **Global** (the ablated rule of §3.3): one pooled test over all uncertified strata —
  $\hat U_{pool} + \psi_{pool} < \varepsilon$ and no hot pending search anywhere. $\hat U_{pool}$ was logged
  live by the production estimator (so frontier credit is counted once); $\psi_{pool}$ is recomputed here
  with $w=1$ (a single test pays no union bound — the global arm gets its *best* case) from the pooled-$N$
  trajectory, accumulating $V_t = \sum c_t^2$, $c_t = Y_{CAP}(1+\beta)/N_t$, exactly as the production
  radius does per stratum.

Each replay runs in both pre-registered variants: **full** (with ψ) and **estimate** (Û only — the worked
run's regime). When a test never fires inside the recorded prefix, that arm inherits the run's own terminal
stop (budget / step-cap / economic), which made no promise.

In [ ]:
# --- the offline stop-rule replays -------------------------------------------
def replay_stratified(df, eps, budget_usd, terminal, with_psi=True):
    """First step at which the per-group rule stops, and why (mirror of
    pipeline.all_strata_pass + the budget guard, in the loop's order)."""
    for step, grp in df.groupby("step", sort=True):
        if float(grp["spent"].max()) >= budget_usd:      # budget checked FIRST,
            return int(step), "budget"                   # as in the live loop
        ok = True
        for row in grp.itertuples():
            if row.certified:                            # checksum-closed: exempt
                continue
            if row.claimed_count is not None and row.N < row.claimed_count:
                ok = False; break                        # cardinality brake
            if not (row.U + (row.psi if with_psi else 0.0) < eps):
                ok = False; break                        # conjunct (i)
            if row.hot:
                ok = False; break                        # conjunct (ii)
        if ok and len(grp) > 0:                          # cold-start guard
            return int(step), "certified"
    return terminal                                      # never fired in prefix

def replay_global(df, dfg, eps, delta, budget_usd, terminal, with_psi=True):
    """First step at which the single pooled test passes, and why. Pooled
    U_hat comes from the live U_hat_global rows; pooled psi is rebuilt here
    (w = 1) from the pooled-N trajectory, mirroring frontier.psi exactly."""
    V = 0.0
    hot_by_step = (df.groupby("step")["hot"].any()
                   if len(df) else pd.Series(dtype=bool))
    for r in dfg.sort_values("step").itertuples():
        step = int(r.step)
        if float(r.spent) >= budget_usd:
            return step, "budget"
        c = Y_CAP * (1 + BETA) / max(int(r.N), 1)        # per-occasion bound c_t
        V += c * c                                       # realized V_t (pooled)
        psi = math.sqrt(2 * V * math.log(1 / delta)) if with_psi else 0.0
        if r.n_strata == 0:
            continue                                     # cold-start guard
        if r.n_uncertified == 0:
            return step, "certified"                     # all checksum-closed
        grp = df[df["step"] == step]
        unc = grp[grp["certified"].isnull()]
        brake = any((row.claimed_count is not None) and (row.N < row.claimed_count)
                    for row in unc.itertuples())         # shared claims machinery
        if (r.Ug + psi < eps) and not bool(hot_by_step.get(step, False)) \
                and not brake:
            return step, "certified"
    return terminal

stop_rows = []
for pid in done:
    df, dfg, term = step_logs[pid], glob_logs[pid], actual_stops[pid]
    for variant, wp in [("full", True), ("estimate", False)]:
        s, why = replay_stratified(df, EPS, BUDGET_USD, term, with_psi=wp)
        stop_rows.append(dict(portfolio=pid, arm="stratified", variant=variant,
                              stop_step=s, reason=why))
        s, why = replay_global(df, dfg, EPS, DELTA, BUDGET_USD, term, with_psi=wp)
        stop_rows.append(dict(portfolio=pid, arm="global", variant=variant,
                              stop_step=s, reason=why))
stops = pd.DataFrame(stop_rows)
spent_at = {p: glob_logs[p].set_index("step")["spent"] for p in done}
stops["spent"] = [float(spent_at[r.portfolio].get(r.stop_step, np.nan))
                  for r in stops.itertuples()]
stops.pivot_table(index="portfolio", columns=["variant", "arm"],
                  values="stop_step", aggfunc="first").astype("Int64")

In [ ]:
# --- SELF-TESTS: the replays must reproduce reality, or nothing else counts --
# 1. the full-variant stratified replay IS the live rule at the live eps:
#    it must land on the run's own stop, step and reason (Exp-2 discipline).
for pid in done:
    got = replay_stratified(step_logs[pid], EPS, BUDGET_USD,
                            actual_stops[pid], with_psi=True)
    assert got == actual_stops[pid], (
        f"replay diverges from reality for {pid}: replay={got}, "
        f"actual={actual_stops[pid]} -- instrumentation or replay logic wrong")

# 2. the patch's pooled row must equal the sum of the per-stratum rows it
#    claims to pool (N and f1, over uncertified strata, at every step)
for pid in done:
    df, dfg = step_logs[pid], glob_logs[pid]
    per = (df[df["certified"].isnull()]
           .groupby("step")[["N", "f1"]].sum())
    for r in dfg.itertuples():
        if r.n_strata == 0 or r.n_uncertified == 0 or r.step not in per.index:
            continue
        assert int(per.loc[r.step, "N"]) == int(r.N) and \
               int(per.loc[r.step, "f1"]) == int(r.f1), (
            f"{pid} step {r.step}: pooled row (N={r.N}, f1={r.f1}) != "
            f"sum of per-stratum rows {tuple(per.loc[r.step])}")

# 3. the offline pooled psi mirrors the production radius exactly: feed the
#    same realized V into frontier.psi with w = 1 and compare
_V = 0.0
for n in (0, 3, 5, 9, 9, 14):
    _c = Y_CAP * (1 + BETA) / max(n, 1)
    _V += _c * _c
_off = math.sqrt(2 * _V * math.log(1 / (DELTA * 1.0)))
_prod = FrontierState(Y_CAP=Y_CAP, BETA=BETA).psi(
    pool=set(), delta_M=DELTA, w_g=1.0, max_occasions=999, V_realized=_V)
assert abs(_off - _prod) < 1e-12, "offline pooled psi != production psi"

# 4. expected ordering (empirical, printed not asserted): the pooled test
#    should fire no later than the per-group test on every portfolio
for variant in ("full", "estimate"):
    sub = stops[stops["variant"] == variant].pivot(
        index="portfolio", columns="arm", values="stop_step")
    late = sub[sub["global"] > sub["stratified"]]
    if len(late):
        print(f"NOTE ({variant}): global stopped LATER than stratified on "
              f"{list(late.index)} -- report as a finding")
print(f"self-tests passed on all {len(done)} portfolios: live stop reproduced, "
      f"pooled rows consistent, psi mirror exact.")

## Step 5 (offline) — grade each stop against the withheld answer key, per company

Three bookkeeping moves turn the flight recorder into per-company recall curves:

1. **Which company does a record belong to?** Strata are pre-ER surface forms ("hinge health", "loft labs");
   each resolved record's contributing mentions carry those surfaces. A deterministic name map (exact match,
   containment, or token-subset against the portfolio's normalized company names; longest name wins ties;
   majority vote across a record's mentions) assigns records and strata to portfolio members. Whatever maps
   to no member is off-portfolio noise — counted and reported, never graded.
2. **When was each record found?** mention → source page → the search that fetched it → the ledger step that
   search ran. A record exists once its *earliest* supporting mention has arrived (Exp-2 convention).
3. **Is it real?** Records are aligned one-to-one with the registry's truth records by the pre-registered
   A3 attempt-#3 grading key (`match_to_truth(..., amount_primary=True)`, 5 % amount tolerance). **Recall
   for company $g$ at step $t$** = the fraction of $g$'s true records whose first evidence had arrived by
   $t$. Pooled recall is the same fraction over the whole portfolio — the only number the global test can
   see.

One approximation, inherited from Exp 2 and stated in Limitations: ER runs once on the full pool and is then
time-sliced, rather than re-run at every prefix.

In [ ]:
# --- name mapping, discovery-step attribution, per-company recall ------------
def stratum_member(g: str, member_norms: dict[str, str]) -> str | None:
    """Deterministic surface -> portfolio-member map (None = off-portfolio)."""
    for eid, nm in member_norms.items():
        if g == nm:
            return eid                                   # exact wins outright
    cands = []
    for eid, nm in member_norms.items():
        gt, nt = set(g.split()), set(nm.split())
        if nm and (nm in g or g in nm or nt <= gt):
            cands.append((len(nm), eid))                 # longest name wins ties
    return max(cands)[1] if cands else None

def load_mention_info(db_path: str, run_id: str, df: pd.DataFrame):
    """mention_id -> (normalized surface, ledger step of its source's search).
    Route: mention -> source -> formulation_id -> earliest ledger step; fall
    back to the formulation's PROPOSAL step (errs early; counted)."""
    fid_step = df.groupby("formulation_id")["step"].min().to_dict()
    session = get_session(db_path)
    try:
        prop = {f.formulation_id: f.step for f in
                session.query(FrontierFormulationRow).filter_by(run_id=run_id)}
        src_fid = {s.source_id: s.formulation_id for s in session.query(SourceRow)}
        rows = [(m.mention_id, m.source_id, m.entity_surface)
                for m in session.query(MentionRow)]
    finally:
        engine = session.get_bind(); session.close(); engine.dispose()
    out, fell_back = {}, 0
    for mid, sid, surf in rows:
        fid = src_fid.get(sid)
        if fid in fid_step:
            step = int(fid_step[fid])
        else:
            step = int(prop.get(fid, 1)) or 1
            fell_back += 1
        out[mid] = (normalize_surface(surf or ""), step)
    if fell_back:
        print(f"    note: {fell_back}/{len(rows)} mentions attributed via "
              f"proposal-step fallback")
    return out

def load_records(pid: str) -> list[dict]:
    """The run's REAL resolved records, reshaped so risk_control's graders
    (dict records whose attribute values expose .value/.value_num) accept them."""
    out = []
    for r in json.loads((EXP_DIR / f"records_{pid}.json").read_text()):
        attrs = {a: (SimpleNamespace(**cv) if cv else None)
                 for a, cv in r["attributes"].items()}
        out.append({"entity_id": r["entity_id"], "record_kind": r["record_kind"],
                    "attributes": attrs,
                    "contributing_mentions": r["contributing_mentions"]})
    return out

records_by, member_of, first_step_of, noise_counts = {}, {}, {}, {}
for pid in done:
    pf = portfolios[pid]
    member_norms = {e: normalize_surface(pf["names"][e]) for e in pf["members"]}
    minfo = load_mention_info(runs_index[pid]["db"], runs_index[pid]["run_id"],
                              step_logs[pid])
    recs, mem, first, noise = load_records(pid), [], [], 0
    for r in recs:
        votes, steps = {}, []
        for mid in r["contributing_mentions"]:
            surf, step = minfo.get(mid, ("", 10**9))
            steps.append(step)
            e = stratum_member(surf, member_norms)
            if e is not None:
                votes[e] = votes.get(e, 0) + 1
        # majority vote over the record's mention surfaces; ties by entity id
        owner = (sorted(votes.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
                 if votes else None)
        noise += owner is None
        mem.append(owner)
        first.append(min(steps) if steps else 10**9)
    records_by[pid], member_of[pid] = recs, mem
    first_step_of[pid], noise_counts[pid] = first, noise
    print(f"{pid}: {len(recs)} resolved records, {noise} off-portfolio (noise)")

# first ledger step at which ANY stratum of each member appears: a member with
# no stratum by step t is INVISIBLE to both stopping rules at t (Theorem 2's
# delta_G channel -- undiscovered strata -- not an eps_g failure)
first_seen: dict[str, dict[str, int]] = {}
for pid in done:
    pf = portfolios[pid]
    member_norms = {e: normalize_surface(pf["names"][e]) for e in pf["members"]}
    fs: dict[str, int] = {}
    for row in step_logs[pid].itertuples():
        e = stratum_member(str(row.stratum), member_norms)
        if e is not None and int(row.step) < fs.get(e, 10**9):
            fs[e] = int(row.step)
    first_seen[pid] = fs

def recall_at(pid: str, eid: str, t: int) -> dict:
    """Record + value recall for one member at step t: records attributed to
    eid whose first evidence arrived by t, graded one-to-one against the
    registry (pre-registered amount-primary key, 5% tolerance)."""
    sub = [r for r, m, s in zip(records_by[pid], member_of[pid],
                                first_step_of[pid]) if m == eid and s <= t]
    truth = truths[eid]
    aligned = match_to_truth(sub, truth, key_fn=default_truth_key,
                             amount_tol=AMOUNT_TOL, amount_primary=True)
    hit = [tr for _, tr in aligned if tr is not None]
    return {"record_recall": len(hit) / max(truth.true_count, 1),
            "value_recall": (sum(x.amount for x in hit) / truth.true_sum
                             if truth.true_sum else float("nan")),
            "n_matched": len(hit), "n_truth": truth.true_count}

def portfolio_recalls(pid: str, t: int) -> dict:
    per = {e: recall_at(pid, e, t) for e in portfolios[pid]["members"]}
    tot_hit = sum(v["n_matched"] for v in per.values())
    tot_truth = sum(v["n_truth"] for v in per.values())
    return {"per_member": {e: v["record_recall"] for e, v in per.items()},
            "pooled": tot_hit / max(tot_truth, 1),
            "small": per[portfolios[pid]["small"]]["record_recall"],
            "worst": min(v["record_recall"] for v in per.values())}

# grade every (portfolio, arm, variant) at its stop step
graded = []
for r in stops.itertuples():
    g = portfolio_recalls(r.portfolio, int(r.stop_step))
    pf = portfolios[r.portfolio]
    undisc = [pf["names"][e] for e in pf["members"]
              if first_seen[r.portfolio].get(e, 10**9) > int(r.stop_step)]
    graded.append(dict(pooled_recall=g["pooled"], small_recall=g["small"],
                       worst_recall=g["worst"],
                       undiscovered_at_stop=json.dumps(undisc),
                       member_recalls=json.dumps(
                           {pf["names"][e]: round(v, 3)
                            for e, v in g["per_member"].items()})))
stops = pd.concat([stops.reset_index(drop=True), pd.DataFrame(graded)], axis=1)

# the pre-registered event definitions
certified = stops["reason"] == "certified"
stops["pooled_promise_kept"] = certified & (stops["pooled_recall"] >= 1 - EPS)
stops["silent_failure"] = ((stops["arm"] == "global") & certified
                           & (stops["pooled_recall"] >= 1 - EPS)
                           & (stops["small_recall"] < 1 - EPS))
stops["violation"] = ((stops["arm"] == "stratified") & certified
                      & (stops["worst_recall"] < 1 - EPS))
stops.round(3)

## Figure 1 — one portfolio's journey (the Cinder failure made visible)

Two synchronized views of the most contrastive portfolio, on the estimate-level variant. **Top — what the
agent believes:** the orange curve is the *pooled* unseen-mass estimate the global rule watches; the blue
dashed curve is the same estimate for the small company alone; the dotted line is the promise ε. **Bottom —
the truth it cannot see:** each grey step-curve is one company's actually-missing fraction, the small company
drawn thick. The orange marker is where the global rule says "done": its pooled curve is under ε — and the
small company below is still missing most of its history. The blue marker is where the stratified rule
finally lets go: only once the small company's own curve has come down. That vertical gap at the orange
marker *is* the silent failure — invisible in the top panel's pooled orange curve, obvious in the bottom
panel.

In [ ]:
# --- Figure 1: hero trajectory of the most contrastive portfolio -------------
est = stops[stops["variant"] == "estimate"]
piv = est.pivot(index="portfolio", columns="arm",
                values=["stop_step", "reason", "small_recall"])
cands = [p for p in done if piv.loc[p, ("reason", "global")] == "certified"]
rep = (max(cands, key=lambda p: piv.loc[p, ("small_recall", "stratified")]
           - piv.loc[p, ("small_recall", "global")]) if cands else done[0])
pf, df, dfg = portfolios[rep], step_logs[rep], glob_logs[rep]
small_eid = pf["small"]
small_norm = normalize_surface(pf["names"][small_eid])
member_norms = {e: normalize_surface(pf["names"][e]) for e in pf["members"]}
t_glob = int(piv.loc[rep, ("stop_step", "global")])
t_strat = int(piv.loc[rep, ("stop_step", "stratified")])
steps_x = sorted(dfg["step"].unique())

fig, (axT, axB) = plt.subplots(2, 1, figsize=(9.2, 6.8), sharex=True,
                               height_ratios=[1, 1.25])
# top: the agent's statistics
axT.plot(dfg["step"], np.minimum(dfg["Ug"], 1.0), color=ARM_COLOR["global"],
         lw=2.2, label=r"pooled estimate $\hat{U}_{pool}$ (what the global rule watches)")
small_rows = df[df["stratum"].map(
    lambda g: stratum_member(str(g), member_norms) == small_eid)]
if len(small_rows):
    su = small_rows.groupby("step")["U"].max()
    axT.plot(su.index, np.minimum(su.values, 1.0), color=ARM_COLOR["stratified"],
             lw=1.8, ls="--",
             label=fr"{pf['names'][small_eid]} alone: $\hat{{U}}_g$")
axT.axhline(EPS, color="#333333", ls=":", lw=1.2)
axT.text(steps_x[-1], EPS, fr"  $\varepsilon$={EPS}", va="center", fontsize=9)
axT.set_ylabel("estimated unseen fraction")
axT.set_ylim(-0.03, 1.05)
axT.legend(fontsize=8.5, loc="upper right")
# bottom: the withheld truth
for e in pf["members"]:
    rc = np.array([recall_at(rep, e, t)["record_recall"] for t in steps_x])
    is_small = e == small_eid
    axB.plot(steps_x, 1 - rc, drawstyle="steps-post",
             color=ARM_COLOR["stratified"] if is_small else "#BBBBBB",
             lw=2.4 if is_small else 1.2,
             label=(f"{pf['names'][e]} (small: "
                    f"{pf['truth_counts'][e]} true records)" if is_small
                    else None), zorder=5 if is_small else 2)
axB.axhline(EPS, color="#333333", ls=":", lw=1.2)
for ax in (axT, axB):
    ax.axvline(t_glob, color=ARM_COLOR["global"], lw=1.2, alpha=0.7)
    ax.axvline(t_strat, color=ARM_COLOR["stratified"], lw=1.2, alpha=0.7)
miss_g = 1 - recall_at(rep, small_eid, t_glob)["record_recall"]
axB.scatter([t_glob], [miss_g], s=110, color=ARM_COLOR["global"],
            edgecolor="k", zorder=6,
            label=f'global says "done": small company still {miss_g:.0%} missing')
miss_s = 1 - recall_at(rep, small_eid, t_strat)["record_recall"]
axB.scatter([t_strat], [miss_s], s=110, color=ARM_COLOR["stratified"],
            edgecolor="k", zorder=6,
            label=f'stratified says "done": {miss_s:.0%} missing')
axB.set_xlabel("agent step (one real search per step)")
axB.set_ylabel("TRUE missing fraction\n(hidden from the agent)")
axB.set_ylim(-0.05, 1.08)
axB.legend(fontsize=8.5, loc="upper right")
axT.set_title(f"Exp 3, Fig. 1 -- portfolio {rep}: "
              f"{', '.join(pf['names'][e] for e in pf['members'])}")
fig.tight_layout()
fig.savefig(FIG_DIR / "exp3_hero_trajectory.pdf", bbox_inches="tight")
print(f"representative portfolio: {rep} "
      f"(global stop {t_glob}, stratified stop {t_strat})")

## Figure 2 — every portfolio: the small company at each rule's stop

One column per portfolio (estimate-level variant). The orange dot is the small company's recall at the
moment the **global** rule stopped; the blue dot is the same company at the **stratified** stop; the
connecting line is what the extra patience bought *that company*. The grey diamond is the **pooled** recall
at the global stop — when it sits above the dashed promise line while the orange dot sits below, that
portfolio is a silent failure in one glance: *the global certificate was true about the pool and wrong about
the group*. Hollow markers mean that arm never certified inside the run (it inherited a budget/step-cap
stop, which made no promise).

In [ ]:
# --- Figure 2: paired small-company recall, global vs stratified stop --------
fig, ax = plt.subplots(figsize=(8.8, 4.8))
order = sorted(done)
for i, pid in enumerate(order):
    row_g = est[(est["portfolio"] == pid) & (est["arm"] == "global")].iloc[0]
    row_s = est[(est["portfolio"] == pid) & (est["arm"] == "stratified")].iloc[0]
    ax.plot([i, i], [row_g["small_recall"], row_s["small_recall"]],
            color="#888888", lw=1.2, zorder=2)
    for row, arm, dx in [(row_g, "global", 0), (row_s, "stratified", 0)]:
        filled = row["reason"] == "certified"
        ax.scatter([i + dx], [row["small_recall"]], s=90,
                   color=ARM_COLOR[arm] if filled else "none",
                   edgecolor=ARM_COLOR[arm], lw=1.6, zorder=5)
    ax.scatter([i], [row_g["pooled_recall"]], marker="D", s=55,
               color="#AAAAAA", edgecolor="k", lw=0.5, zorder=4)
ax.axhline(1 - EPS, color="#333333", ls="--", lw=1.4)
ax.text(len(order) - 0.5, 1 - EPS, f"  promise $\\geq$ {1-EPS:.2f}",
        va="bottom", fontsize=9)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{p}\n{portfolios[p]['names'][portfolios[p]['small']]}"
                    for p in order], fontsize=8.5)
ax.set_ylabel("small company's record recall at the stop")
ax.set_ylim(-0.05, 1.1)
handles = [plt.Line2D([], [], marker="o", ls="", color=ARM_COLOR["global"],
                      label="at the GLOBAL stop"),
           plt.Line2D([], [], marker="o", ls="", color=ARM_COLOR["stratified"],
                      label="at the STRATIFIED stop"),
           plt.Line2D([], [], marker="D", ls="", color="#AAAAAA",
                      markeredgecolor="k", label="pooled recall at the global stop"),
           plt.Line2D([], [], marker="o", ls="", markerfacecolor="none",
                      color="#555555", label="hollow = no certificate (no promise)")]
ax.legend(handles=handles, fontsize=8.5, loc="lower right")
ax.set_title("Exp 3, Fig. 2 -- the global certificate can be true about the pool "
             "and wrong about the group")
fig.tight_layout()
fig.savefig(FIG_DIR / "exp3_paired_recall.pdf", bbox_inches="tight")
plt.show()

## Figure 3 — the price of the per-group guarantee

The stratified rule's honesty is not free: it keeps searching after the pooled signal has gone quiet. Per
portfolio: search steps (left) and search spend (right) at each rule's stop. This is the cost side of the
trade the paper asks the reader to accept — a few extra dollars of patience versus a silently absent
company. (Search spend only; LLM cost scales with the same step counts.)

In [ ]:
# --- Figure 3: steps and spend at each arm's stop ----------------------------
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10.5, 4.2))
x = np.arange(len(order))
for ax, col, ylab in [(axL, "stop_step", "steps at stop"),
                      (axR, "spent", "search spend at stop (USD)")]:
    for j, arm in enumerate(["global", "stratified"]):
        vals = [est[(est["portfolio"] == p) & (est["arm"] == arm)].iloc[0][col]
                for p in order]
        ax.bar(x + (j - 0.5) * 0.38, vals, width=0.36, color=ARM_COLOR[arm],
               label=arm if ax is axL else None)
    ax.set_xticks(x); ax.set_xticklabels(order)
    ax.set_ylabel(ylab)
axL.legend(fontsize=9)
fig.suptitle("Exp 3, Fig. 3 -- what the per-group guarantee costs", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "exp3_cost.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# --- tables: the paper's numbers ---------------------------------------------
stop_table = stops.copy()
stop_table["small_company"] = stop_table["portfolio"].map(
    lambda p: portfolios[p]["names"][portfolios[p]["small"]])
stop_table = stop_table[["portfolio", "small_company", "arm", "variant",
                         "stop_step", "reason", "spent", "pooled_recall",
                         "small_recall", "worst_recall", "pooled_promise_kept",
                         "silent_failure", "violation", "undiscovered_at_stop",
                         "member_recalls"]]
stop_table.to_csv(EXP_DIR / "exp3_stop_table.csv", index=False)

summary = (stops.groupby(["variant", "arm"])
           .apply(lambda g: pd.Series({
               "n_portfolios":     len(g),
               "n_certified":      int((g["reason"] == "certified").sum()),
               "n_silent_failures": int(g["silent_failure"].sum()),
               "silent_failure_rate": g["silent_failure"].sum()
                   / max((g["reason"] == "certified").sum(), 1),
               "n_violations":     int(g["violation"].sum()),
               "delta_allowed":    DELTA,
               "mean_stop_step":   g["stop_step"].mean(),
               "mean_spend_usd":   g["spent"].mean(),
               "mean_pooled_recall": g["pooled_recall"].mean(),
               "mean_small_recall":  g["small_recall"].mean(),
               "mean_worst_recall":  g["worst_recall"].mean()}),
                  include_groups=False)
           .reset_index())
summary.to_csv(EXP_DIR / "exp3_summary.csv", index=False)
print(f"wrote {EXP_DIR/'exp3_stop_table.csv'}\nwrote {EXP_DIR/'exp3_summary.csv'}")
summary.round(3)

## Reading the results — the pre-registered verdicts

**C3, the Cinder failure (Figures 1–2, `estimate` rows of the summary).** *Pass:* on a substantial fraction
of portfolios the global arm certifies with the pooled promise **kept** (grey diamond above the line) while
the small company sits **below** it — `silent_failure_rate` is the headline number, and the paper's §3.4
prediction ("silently ~25 % wrong") is checked against `1 − small_recall` at the global stops. *Pass, other
half:* among **stratified** certified stops, every member meets the target simultaneously — `n_violations`
compared against δ (Theorem 2's rate). *Honest bad outcome (proposal):* if the global arm rarely certifies
early, or rarely leaves the small company short, stratification solves a rare problem on this cohort — that
is reportable as-is, with the cost columns quantifying what stratification would still charge.

**The `full` variant (production test, with ψ).** Expected: **neither arm ever certifies** at this cohort's
scale — every row inherits its run's terminal stop. This is not a bug to hide but the paper's own remark
made empirical (Thm 2: at small $N_g$ "the statistical certificate may be unattainable at any sane budget —
the truthful state of affairs, not a defect"; tiny strata are meant to close via claims or registries). One
sentence in the evaluation section, next to the estimate-level results.

**Undiscovered members (the δ_G channel).** A company whose stratum did not yet *exist* at a stop is
invisible to **both** rules — no stopping test can examine a stratum it has never seen. That is Theorem 2's
separate δ_G term (controlled by certifying the stratum enumeration or a cardinality claim, not by ε_g), and
the `undiscovered_at_stop` column makes every such event auditable: a stratified "violation" whose short
member appears in that column is a δ_G event, not an ε_g failure — report the two separately. The raw
`violation` flag stays deliberately un-adjusted (pre-registered as-is); the split happens in the write-up,
with the column as evidence.

### Limitations (stated, not hidden)

- **The estimate-level variant is a documented deviation** from the production stop test, pre-registered
  above with its rationale (the worked run's own regime) and always reported *alongside* the full test.
- **ER is computed once on the full pool and time-sliced** by first-evidence step (Exp-2 convention); early
  stop points are graded with cluster assignments the run only settled later.
- **Small denominators.** A "Cinder" holds 2–3 true records, so its recall moves in steps of 1/2 or 1/3 —
  per-portfolio numbers are coarse by design; the cross-portfolio rates in the summary are the statistics
  to quote.
- **Name-based attribution.** Strata are pre-ER surface forms mapped to companies deterministically;
  off-portfolio noise records are excluded and counted (`noise_counts`), unmapped strata never grade.
- **Pre-registered exclusions.** The 7 generic-name-starved entities are excluded from membership (they
  test findability, δ_G, not stopping); the validation half stays sealed. Both are one-line supervisor
  items, filed before any live run.